# 初步实验结果分析

**状态：初步结果，存在问题待解决**

模型对比：Mean 基线 vs LSTM（论文超参数）
- 窗口：(24, 24) — 过去 24h 预测未来 24h
- 数据：institutions/agg_1_hour/n_bytes（283 条序列）
- LSTM 超参数：bidirectional, hidden=100, lr=0.01, epochs=100, patience=5

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial']
matplotlib.rcParams['axes.unicode_minus'] = False

from pathlib import Path
import glob

OUTPUT_DIR = Path('../experiments/02_analysis/figures')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 加载最新结果
lstm_file = sorted(glob.glob('../experiments/results/dl_results_LSTM_*.csv'))[-1]
mean_file = sorted(glob.glob('../experiments/results/baseline_results_*.csv'))[-1]

lstm = pd.read_csv(lstm_file)
mean_ = pd.read_csv(mean_file)

print(f'LSTM 结果: {os.path.basename(lstm_file)}')
print(f'Mean 结果: {os.path.basename(mean_file)}')

## 1. 过滤器：只保留 institutions（283 条）

In [ ]:
lstm_i = lstm[lstm.TS_GROUP == 'institutions'].copy()
mean_i = mean_[mean_.TS_GROUP == 'institutions'].copy()

print(f'LSTM: {len(lstm_i)} records')
print(f'Mean: {len(mean_i)} records')

# 合并用于对比
comp = lstm_i.merge(mean_i, on='TS_ID', suffixes=('_lstm', '_mean'))
print(f'合并后: {len(comp)} records')

## 2. 整体对比

In [ ]:
print('=' * 60)
print('Mean 基线')
print('=' * 60)
print(f'  RMSE: mean={mean_i.RMSE.mean():.2e}  median={mean_i.RMSE.median():.2e}')
print(f'  R²:   mean={mean_i.R2_SCORE.mean():.4f}  median={mean_i.R2_SCORE.median():.4f}')
print(f'  sMAPE: mean={mean_i.SMAPE.mean():.2f}%')
print()
print('=' * 60)
print('LSTM（论文超参数）')
print('=' * 60)
print(f'  RMSE: mean={lstm_i.RMSE.mean():.2e}  median={lstm_i.RMSE.median():.2e}')
print(f'  R²:   mean={lstm_i.R2_SCORE.mean():.4f}  median={lstm_i.R2_SCORE.median():.4f}')
print(f'  sMAPE: mean={lstm_i.SMAPE.mean():.2f}%')
print()

better = (comp.R2_SCORE_lstm > comp.R2_SCORE_mean).sum()
print(f'LSTM 优于 Mean 的序列: {better}/{len(comp)} ({better/len(comp)*100:.1f}%)')

## 3. 发现严重问题：部分序列 R² 极端异常

R² 均值被几条极端异常的序列拉崩了。

In [ ]:
r2 = lstm_i.R2_SCORE
print('R² 分布：')
print(f'  min: {r2.min():.2e}')
print(f'  1%%: {r2.quantile(0.01):.2e}')
print(f'  5%%: {r2.quantile(0.05):.4f}')
print(f'  25%%: {r2.quantile(0.25):.4f}')
print(f'  **中位数: {r2.median():.4f}')
print(f'  75%%: {r2.quantile(0.75):.4f}')
print(f'  95%%: {r2.quantile(0.95):.4f}')
print()

print('R² 分布区间统计：')
bins = [-1e10, -1000, -10, -1, 0, 0.1, 0.3, 0.5, 1.0]
labels = ['<-1000', '-1000~-10', '-10~-1', '-1~0', '0~0.1', '0.1~0.3', '0.3~0.5', '0.5~1']
r2_binned = pd.cut(r2, bins=bins, labels=labels)
print(r2_binned.value_counts().sort_index())

### 3.1 根因分析：数据分布漂移

异常序列的共同特征：**测试集流量骤降**（训练集活跃 → 测试集归零）

时间分割：训练集 (2023-10 ~ 2024-04) → 验证集 → 测试集 (2024-06 ~ 2024-07)

部分机构（如学校）的流量在 6~7 月（暑假）接近归零，而模型学习的是活跃期模式。

In [ ]:
from src.preprocessing import load_and_align, impute_missing

def show_split_stats(file_id):
    df = load_and_align(file_id)
    df = impute_missing(df, 'zeros')
    n = len(df)
    train = df.iloc[:int(n*0.7)]
    test = df.iloc[int(n*0.85):]
    return {
        'train_mean': train['n_bytes'].mean(),
        'test_mean': test['n_bytes'].mean(),
        'ratio': test['n_bytes'].mean() / max(train['n_bytes'].mean(), 1),
        'train_period': f"{train.index[0].strftime('%Y-%m')}~{train.index[-1].strftime('%Y-%m')}",
        'test_period': f"{test.index[0].strftime('%Y-%m')}~{test.index[-1].strftime('%Y-%m')}",
    }

print('=== 最差 3 条序列的分布漂移 ===')
worst3 = lstm_i.nsmallest(3, 'R2_SCORE')
for _, row in worst3.iterrows():
    s = show_split_stats(int(row['TS_ID']))
    print(f"  file={int(row['TS_ID']):>3d} | R²={row['R2_SCORE']:>10.2f} | "
          f"train_mean={s['train_mean']:>12.0f} | test_mean={s['test_mean']:>8.0f} | "
          f"ratio={s['ratio']:.4f}")

print()
print('=== 最好 3 条序列的分布对比 ===')
best3 = lstm_i.nlargest(3, 'R2_SCORE')
for _, row in best3.iterrows():
    s = show_split_stats(int(row['TS_ID']))
    print(f"  file={int(row['TS_ID']):>3d} | R²={row['R2_SCORE']:>10.4f} | "
          f"train_mean={s['train_mean']:>12.0f} | test_mean={s['test_mean']:>12.0f} | "
          f"ratio={s['ratio']:.4f}")

## 4. 修正评估：剔除极端异常序列后

参考论文做法，对 R² 做裁剪（clip），只看合理范围内的序列。

In [ ]:
# 剔除 R² < -10 的极端异常序列，做二次统计
mask = lstm_i.R2_SCORE > -10

lstm_clean = lstm_i[mask]
mean_clean = mean_i[mask]

print(f'剔除前: {len(lstm_i)} 条')
print(f'剔除后: {len(lstm_clean)} 条（剔除了 {(~mask).sum()} 条极端异常）')
print()

print('=' * 60)
print('Mean 基线（剔除后）')
print('=' * 60)
print(f'  RMSE: median={mean_clean.RMSE.median():.2e}')
print(f'  R²:   mean={mean_clean.R2_SCORE.mean():.4f}  median={mean_clean.R2_SCORE.median():.4f}')
print()
print('=' * 60)
print('LSTM（剔除后）')
print('=' * 60)
print(f'  RMSE: median={lstm_clean.RMSE.median():.2e}')
print(f'  R²:   mean={lstm_clean.R2_SCORE.mean():.4f}  median={lstm_clean.R2_SCORE.median():.4f}')

## 5. 可视化：R² 分布对比

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 左：完整分布（含异常值）
ax = axes[0]
ax.hist(np.clip(lstm_i.R2_SCORE, -10, 1), bins=50, alpha=0.7, label='LSTM')
ax.hist(np.clip(mean_i.R2_SCORE, -10, 1), bins=50, alpha=0.5, label='Mean')
ax.axvline(0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('R² (clipped to [-10, 1])')
ax.set_ylabel('Count')
ax.set_title('R² Distribution (clipped)')
ax.legend()
ax.grid(True, alpha=0.3)

# 右：去掉极端值后的对比直方图
ax = axes[1]
ax.hist(lstm_clean.R2_SCORE, bins=40, alpha=0.7, label=f'LSTM (n={len(lstm_clean)})')
ax.hist(mean_clean.R2_SCORE, bins=40, alpha=0.5, label=f'Mean (n={len(mean_clean)})')
ax.axvline(0, color='red', linestyle='--', alpha=0.5)
ax.axvline(lstm_clean.R2_SCORE.median(), color='blue', linestyle=':', label=f'LSTM median={lstm_clean.R2_SCORE.median():.3f}')
ax.axvline(mean_clean.R2_SCORE.median(), color='orange', linestyle=':', label=f'Mean median={mean_clean.R2_SCORE.median():.3f}')
ax.set_xlabel('R²')
ax.set_ylabel('Count')
ax.set_title('R² Distribution (outliers removed)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_r2_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'已保存: {OUTPUT_DIR / "01_r2_comparison.png"}')

In [ ]:
# RMSE 分布对比
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (label, data) in zip(axes, [
    ('All data', (lstm_i.RMSE, mean_i.RMSE)),
    ('Outliers removed', (lstm_clean.RMSE, mean_clean.RMSE)),
]):
    ax.hist(np.log10(data[0] + 1), bins=40, alpha=0.7, label='LSTM')
    ax.hist(np.log10(data[1] + 1), bins=40, alpha=0.5, label='Mean')
    ax.set_xlabel('log10(RMSE + 1)')
    ax.set_ylabel('Count')
    ax.set_title(f'RMSE Distribution - {label}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_rmse_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'已保存: {OUTPUT_DIR / "02_rmse_comparison.png"}')

## 6. 逐序列对比：LSTM vs Mean

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

comp_clean = comp[(comp.R2_SCORE_lstm > -10) & (comp.R2_SCORE_mean > -10)]

colors = ['green' if r_l > r_m else 'red' for r_l, r_m in zip(comp_clean.R2_SCORE_lstm, comp_clean.R2_SCORE_mean)]
ax.scatter(comp_clean.R2_SCORE_mean, comp_clean.R2_SCORE_lstm, c=colors, alpha=0.5, s=20)

# 对角线
lim = [-1, 1]
ax.plot(lim, lim, 'k--', alpha=0.3, label='y=x (no improvement)')

ax.set_xlabel('Mean R²')
ax.set_ylabel('LSTM R²')
ax.set_title('Per-series R²: LSTM vs Mean')
ax.legend()
ax.grid(True, alpha=0.3)

# 加统计
above = (comp_clean.R2_SCORE_lstm > comp_clean.R2_SCORE_mean).sum()
below = (comp_clean.R2_SCORE_lstm <= comp_clean.R2_SCORE_mean).sum()
ax.text(0.05, 0.95, f'LSTM better: {above}\nMean better: {below}',
        transform=ax.transAxes, verticalalignment='top', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_per_series_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'已保存: {OUTPUT_DIR / "03_per_series_comparison.png"}')

## 7. 问题总结

### 已确认的问题

1. **数据分布漂移**
   - 部分机构（~20 条）的测试集流量相比训练集骤降 100~100,000 倍
   - 根因：时间分割后测试集落在 6~7 月（暑假），部分机构季节性变化大
   - 影响：这些序列的 R² 极端负值（-1e5 ~ -1e9）严重扭曲平均指标

2. **当前模型效果有限**
   - LSTM 在 63.3% 的序列上优于 Mean 基线
   - 但中位数 R² 仅 0.03，说明大部分序列预测能力很弱
   - 可能原因：lr=0.01 偏大导致早停过快，或模型容量不够

3. **评估指标尺度问题**
   - RMSE 为原始字节数，大机构和小机构之间的误差量级相差极大
   - 直接平均 RMSE 会被大机构主导

### 待尝试的改进方向

- 用 `MinMaxScaler` 替代 `Z-score` 看能否减轻分布漂移的影响
- 降低学习率（lr=0.001）让模型更充分训练
- 增加 patience（10→15）避免过早停止
- 跑更多窗口配置（168,24 等）看趋势是否一致

In [ ]:
print('分析完成。报告输出目录:', OUTPUT_DIR)